# Project Description

We work for the OilyGiant mining company. Our task is to find the best place for a new well. We have data on oil samples from three regions. Parameters of each oil well in the region are already known. 

We need to build a model that will help to pick the region with the highest profit margin. Analyze potential profit and risks using the Bootstrapping technique.

## Conditions:

    - Only linear regression is suitable for model training (the rest are not sufficiently predictable).
    - When exploring the region, a study of 500 points is carried with picking the best 200 points for the profit calculation.
    - The budget for development of 200 oil wells is 100 USD million.
    - One barrel of raw materials brings 4.5 USD of revenue The revenue from one unit of product is 4,500 dollars (volume of reserves is in thousand barrels).
    - After the risk evaluation, keep only the regions with the risk of losses lower than 2.5%. From the ones that fit the    criteria, the region with the highest average profit should be selected.

Let's begin by importing all necessary libraries and dataesetsinto our notebook

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

from scipy import stats

from numpy.random import RandomState

import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
geo_0 = pd.read_csv('/datasets/geo_data_0.csv')
geo_1 = pd.read_csv('/datasets/geo_data_1.csv')
geo_2 = pd.read_csv('/datasets/geo_data_2.csv')

In [3]:
geo_0.info()
print()
geo_1.info()
print()
geo_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Nul

In [4]:
geo_0_dup = geo_0.duplicated().sum()
geo_1_dup = geo_1.duplicated().sum()
geo_2_dup = geo_2.duplicated().sum()

print(f'Geo_0 duplicates: {geo_0_dup}')
print(f'Geo_1 duplicates: {geo_1_dup}')
print(f'Geo_2 duplicates: {geo_2_dup}')

Geo_0 duplicates: 0
Geo_1 duplicates: 0
Geo_2 duplicates: 0


Since there are no missing values or duplicates, let's proceed with data preprocessing and model training. 

## On to the next part of the analysis:

Train and test the model for each region:

 2.1. Split the data into a training set and validation set at a ratio of 75:25.

 2.2. Train the model and make predictions for the validation set.

 2.3. Save the predictions and correct answers for the validation set.

 2.4. Print the average volume of predicted reserves and model RMSE.

 2.5. Analyze the results.

In [5]:
def train_and_evaluate_model(df):
    # Split the data into features and target
    features = df[['f0', 'f1', 'f2']] 
    target = df['product'] 
    
    # Split the data into training (75%) and validation (25%) sets
    features_train, features_val, target_train, target_val = train_test_split(features, target, test_size=0.25, random_state=12345)
    
    # Initialize the linear regression model
    model = LinearRegression()
    
    # Train the model
    model.fit(features_train, target_train)
    
    # Make predictions on the validation set
    predictions_val = model.predict(features_val)
    
    # Save predictions and actual values for the validation set
    results = pd.DataFrame({'Actual': target_val, 'Predicted': predictions_val})
    
    # Calculate average predicted reserves and RMSE
    avg_predicted_reserves = np.mean(predictions_val)
    rmse = np.sqrt(mean_squared_error(target_val, predictions_val))
    
    # Print the results
    print(f"Average Predicted Reserves: {avg_predicted_reserves:.2f}")
    print(f"RMSE: {rmse:.2f}")
    
    return results, avg_predicted_reserves, rmse

# Apply the function to each region
results_0, avg_pred_0, rmse_0 = train_and_evaluate_model(geo_0)
results_1, avg_pred_1, rmse_1 = train_and_evaluate_model(geo_1)
results_2, avg_pred_2, rmse_2 = train_and_evaluate_model(geo_2)

# Optional: Display the results for each region
print("Results for geo_data_0:")
print(results_0.head())
print("\nResults for geo_data_1:")
print(results_1.head())
print("\nResults for geo_data_2:")
print(results_2.head())


Average Predicted Reserves: 92.59
RMSE: 37.58
Average Predicted Reserves: 68.73
RMSE: 0.89
Average Predicted Reserves: 94.97
RMSE: 40.03
Results for geo_data_0:
           Actual  Predicted
71751   10.038645  95.894952
80493  114.551489  77.572583
2655   132.603635  77.892640
53233  169.072125  90.175134
91141  122.325180  70.510088

Results for geo_data_1:
          Actual  Predicted
71751  80.859783  82.663314
80493  53.906522  54.431786
2655   30.132364  29.748760
53233  53.906522  53.552133
91141   0.000000   1.243856

Results for geo_data_2:
           Actual   Predicted
71751   61.212375   93.599633
80493   41.850118   75.105159
2655    57.776581   90.066809
53233  100.053761  105.162375
91141  109.897122  115.303310


Key Takeaways:

- geo_data_1 is the most reliable region for prediction due to its very low RMSE (0.89), meaning the model's predictions are nearly spot-on.


- geo_data_0 and geo_data_2 have higher average reserves (92.59 and 94.97), but their high RMSE (37.58, 40.03) means their predictions are unreliable.


- Although geo_data_1 has the lowest average predicted reserves, it is the most predictable and stable region, which is crucial for making profitable and low-risk investments.

## Prepare for profit calculation:

- 3.1. Store all key values for calculations in separate variables.


- 3.2. Calculate the volume of reserves sufficient for developing a new well without losses. Compare the obtained value with the average volume of reserves in each region.


- 3.3. Provide the findings about the preparation for profit calculation step

In [6]:
budget = 100_000_000  
num_wells = 200  
price_per_barrel = 4.5
unit_price = price_per_barrel * 1000  # Revenue per 1000 barrels of oil

# Calculate cost per well
cost_per_well = budget / num_wells

# Calculate minimum reserves required per well to break even
min_reserves_required = cost_per_well / unit_price

print(f"Minimum reserves required per well to break even: {min_reserves_required:.2f} thousand barrels")
print()
# Print comparison of required reserves vs. predicted reserves
print(f"geo_data_0 - Average Predicted Reserves: {avg_pred_0:.2f} thousand barrels")
print(f"geo_data_1 - Average Predicted Reserves: {avg_pred_1:.2f} thousand barrels")
print(f"geo_data_2 - Average Predicted Reserves: {avg_pred_2:.2f} thousand barrels")

Minimum reserves required per well to break even: 111.11 thousand barrels

geo_data_0 - Average Predicted Reserves: 92.59 thousand barrels
geo_data_1 - Average Predicted Reserves: 68.73 thousand barrels
geo_data_2 - Average Predicted Reserves: 94.97 thousand barrels


Key Takeaways:


- None of the regions have an average predicted reserve that meets or exceeds the break-even threshold (111.11 thousand barrels).


- The highest average predicted reserves are in geo_data_2 (94.97), but this is still below the required 111.11.
- This suggests that if we were to develop all wells in any region, we would likely not break even.

## Write a function to calculate profit from a set of selected oil wells and model predictions:

- 4.1. Pick the wells with the highest values of predictions. 


- 4.2. Summarize the target volume of reserves in accordance with these predictions


- 4.3. Provide findings: suggest a region for oil wells' development and justify the choice. Calculate the profit for the obtained volume of reserves.

In [7]:
def calculate_profit(df, predictions):
    """Calculate profit for the top 200 wells based on highest predicted reserves"""
    
    # Add predictions to the DataFrame
    df['Predicted'] = predictions
    
    # Select the top 200 wells with the highest predicted reserves
    top_wells = df.nlargest(200, 'Predicted')
    
    # Sum actual reserves of these selected wells
    total_reserves = top_wells['product'].sum()  # In thousand barrels
    
    # Calculate revenue
    revenue = total_reserves * unit_price # Convert to USD
    
    # Calculate profit
    profit = revenue - budget  # Subtract the budget
    
    return total_reserves, revenue, profit

# Calculate profit for each region
total_reserves_0, revenue_0, profit_0 = calculate_profit(geo_0, results_0['Predicted'])
total_reserves_1, revenue_1, profit_1 = calculate_profit(geo_1, results_1['Predicted'])
total_reserves_2, revenue_2, profit_2 = calculate_profit(geo_2, results_2['Predicted'])

# Print results
print(f"geo_data_0 - Total Reserves: {total_reserves_0:.2f} thousand barrels | Revenue: ${revenue_0:,.2f} | Profit: ${profit_0:,.2f}")
print(f"geo_data_1 - Total Reserves: {total_reserves_1:.2f} thousand barrels | Revenue: ${revenue_1:,.2f} | Profit: ${profit_1:,.2f}")
print(f"geo_data_2 - Total Reserves: {total_reserves_2:.2f} thousand barrels | Revenue: ${revenue_2:,.2f} | Profit: ${profit_2:,.2f}")

geo_data_0 - Total Reserves: 29601.84 thousand barrels | Revenue: $133,208,260.43 | Profit: $33,208,260.43
geo_data_1 - Total Reserves: 27589.08 thousand barrels | Revenue: $124,150,866.97 | Profit: $24,150,866.97
geo_data_2 - Total Reserves: 28245.22 thousand barrels | Revenue: $127,103,499.64 | Profit: $27,103,499.64


# Choosing the Best Region for Development (based on Model Predictions)

We have the following profit results:

- geo_data_0: $33.21M profit

- geo_data_1: $24.15M profit

- geo_data_2: $27.10M profit

From a pure profit standpoint, geo_data_0 looks like the best choice.

But let's bring back RMSE (model accuracy):

- geo_data_0 RMSE: 37.58 (High error)
- geo_data_1 RMSE: 0.89 (Very low error | Best accuracy)
- geo_data_2 RMSE: 40.03 (High error)

Final Recommendations:

Region 1 (geo_data_1) should be the preferred region for development because:

- Most accurate model (RMSE = 0.89) → Reliable predictions.
- Decent profit ($24.15M) → While not the highest, it’s still profitable.
- Lower risk → Since predictions are more precise, investment decisions will be more stable.

## Calculate risks and profit for each region:

    5.1. Use the bootstrapping technique with 1000 samples to find the distribution of profit.
    

    5.2. Find average profit, 95% confidence interval and risk of losses. Loss is negative profit, calculate it as a probability and then express as a percentage.
    
    
    5.3. Provide findings: suggest a region for development of oil wells and justify the choice.

In [8]:
import numpy as np
import pandas as pd

# Ensure all variables are correctly defined
num_wells = 200  # Selecting top 200 wells
sample_size = 500  # Bootstrapping sample size
n_samples = 1000  # Number of bootstrap samples
unit_price = 4.5 * 1000  # Revenue per 1000 barrels
budget = 100_000_000  # Development budget

# Function to calculate profit
def calculate_profit(predicted, actual, wells_production=num_wells):
    """Calculate profit based on top wells selected by predicted reserves."""
    
    # Create DataFrame for sorting
    combined = pd.DataFrame({"predictions": predicted, "actual": actual})
    
    # Select top wells based on predicted values
    top_wells = combined.sort_values(by="predictions", ascending=False).head(wells_production)

    # Calculate total reserves based on actual values
    total_reserves = top_wells["actual"].sum()  # In thousand barrels
    profit = total_reserves * unit_price - budget  # Compute profit

    return profit

# Function for bootstrapping
def bootstrap_profit(predictions: np.ndarray, actual: np.ndarray, n_samples=1000, sample_size=500):
    """Perform bootstrapping to find profit distribution."""
    
    profits = []
    
    # Convert inputs to DataFrame
    data = pd.DataFrame({"predictions": predictions, "actual": actual})

    for _ in range(n_samples):
        # Sample 500 wells with replacement
        sample = data.sample(n=sample_size, replace=True, random_state=None)
        
        # Calculate profit for the sampled wells
        sample_profit = calculate_profit(sample["predictions"].values, sample["actual"].values)
        profits.append(sample_profit)

    profits = np.array(profits)

    # Compute statistics
    lower, upper = np.percentile(profits, [2.5, 97.5])
    loss_risk = np.mean(profits < 0) * 100  # Probability of loss

    return np.mean(profits), lower, upper, loss_risk

# Ensure predictions and actual values are correctly extracted
pred_0, actual_0 = results_0["Predicted"].values, results_0["Actual"].values
pred_1, actual_1 = results_1["Predicted"].values, results_1["Actual"].values
pred_2, actual_2 = results_2["Predicted"].values, results_2["Actual"].values

# Calculate profit for each region using top 200 wells
profit_0 = calculate_profit(pred_0, actual_0)
profit_1 = calculate_profit(pred_1, actual_1)
profit_2 = calculate_profit(pred_2, actual_2)

print(f"Profit for Region 0: ${profit_0:,.2f}")
print(f"Profit for Region 1: ${profit_1:,.2f}")
print(f"Profit for Region 2: ${profit_2:,.2f}")

# Perform bootstrapping for each region
for i, (pred, actual) in enumerate([(pred_0, actual_0), (pred_1, actual_1), (pred_2, actual_2)]):
    mean_profit, lower, upper, loss_risk = bootstrap_profit(pred, actual)
    
    print(f"\nRegion {i}:")
    print(f"  Average profit: ${mean_profit:,.2f}")
    print(f"  95% confidence interval: ${lower:,.2f} to ${upper:,.2f}")
    print(f"  Risk of loss: {loss_risk:.2f}%")

Profit for Region 0: $33,208,260.43
Profit for Region 1: $24,150,866.97
Profit for Region 2: $27,103,499.64

Region 0:
  Average profit: $3,984,605.01
  95% confidence interval: $-1,121,826.70 to $9,746,804.62
  Risk of loss: 7.10%

Region 1:
  Average profit: $4,551,119.22
  95% confidence interval: $709,709.15 to $8,473,476.77
  Risk of loss: 1.20%

Region 2:
  Average profit: $3,792,413.84
  95% confidence interval: $-1,916,789.13 to $9,509,947.68
  Risk of loss: 8.40%


# Conclusion

Final Recommendation: Choose Region 1


Region 1 is the best choice for oil well development because:


    1. It has the lowest risk (1.2%), meaning it is the safest investment.
    2. The confidence interval is positive, suggesting profitability is highly likely.
    3. The average profit (4.55M) is slightly higher than Region 0 (3.98M), but with much lower risk.

While Region 0 has the highest total profit, it also has significantly more risk (7.1%), making Region 1 the smarter long-term investment.